# AI工学101 — 第24回

## モデル比較と評価設計：Accuracyだけでモデルを選んでいいのか？

第23回では **Gradient Boosting** を学びました。

ここまでで、

```text
Linear Regression
Logistic Regression
Decision Tree
Random Forest
Gradient Boosting
```

と、かなり種類が増えてきました。

そして今、機械学習エンジニアとして重要な問題にぶつかります。

> **結局、どのモデルを採用すればいいの？**

ここで、

```text
Accuracyが一番高いモデル！
```

と決めてしまうのは危険です。

なぜなら、**「何を良い予測とするか」は問題によって違う**からです。

今日は、モデルそのものではなく、

> **モデルをどう評価し、どう比較し、どう採用するか**

を学びます。

---

# 🎯 今日のゴール

今日は、

* Accuracyの限界を理解する
* Precision / Recall / F1を使い分ける
* ROC-AUCの意味を理解する
* Confusion Matrixを読める
* `cross_validate()` を使える
* 複数モデルを同じ条件で比較できる
* 「評価指標そのものが設計対象」という考え方を身につける

ことを目指します。

---

# 📖 講義：約20分

## 1. Accuracyは本当に公平？

例えば1000人の患者について、

```text
病気       10人
健康      990人
```

だったとします。

ここで、

> 「全員健康です」

と予測するAIを作ったとします。

すると、

```text
990 / 1000 = 99%
```

です。

Accuracyはなんと**99%**。

でも、

```text
病気の10人
```

を全員見逃しています。

医療検査だったら、かなり困ります。

つまり、

> **Accuracyが高いことと、役に立つモデルであることは同じではない。**

---

# 📖 2. Confusion Matrix

そこで、予測の中身を分解します。

二値分類なら、

|            | 実際Positive | 実際Negative |
| ---------- | ---------: | ---------: |
| 予測Positive |         TP |         FP |
| 予測Negative |         FN |         TN |

です。

---

## TP — True Positive

```text
実際：病気
予測：病気
```

正しく陽性を当てた。

---

## TN — True Negative

```text
実際：健康
予測：健康
```

正しく陰性を当てた。

---

## FP — False Positive

```text
実際：健康
予測：病気
```

健康なのに「病気」と判断。

**偽陽性**です。

---

## FN — False Negative

```text
実際：病気
予測：健康
```

病気なのに見逃した。

**偽陰性**です。

---

# 💻 実習1：Confusion Matrixを作る

Irisは3クラスなので、今回は二値分類用にデータを作ります。

```python
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target
```

確認。

```python
print(X.shape)
print(y.shape)
```

---

## train/test分割

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習2：Logistic Regression

今回はPipelineを使います。

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

学習。

```python
model.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = model.predict(
    X_test
)
```

---

# 💻 実習3：Accuracy

```python
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    pred
)

print(accuracy)
```

まずはこれ。

---

# 💻 実習4：Confusion Matrix

```python
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    pred
)

print(cm)
```

例えば、

```text
[[40  2]
 [ 3 69]]
```

のような行列が出ます。

この数字を、

```text
TN FP
FN TP
```

として読みます。

ただし、`sklearn` のクラスラベルの並びに依存するので、

> **「左上が必ず病気」などと暗記しない**

こと。

ラベルの意味を確認してから解釈します。

---

# 💻 実習5：Precision

```python
from sklearn.metrics import precision_score

precision = precision_score(
    y_test,
    pred
)

print(precision)
```

Precisionは、

[
Precision =
\frac{TP}{TP+FP}
]

です。

直感的には、

> **「陽性だと言ったもののうち、本当に陽性だった割合」**

です。

---

# 💻 実習6：Recall

```python
from sklearn.metrics import recall_score

recall = recall_score(
    y_test,
    pred
)

print(recall)
```

Recallは、

[
Recall =
\frac{TP}{TP+FN}
]

です。

直感的には、

> **「本当に陽性だったものを、どれだけ拾えたか」**

です。

---

# 🧠 PrecisionとRecallの違い

ここは今日の最重要ポイントの一つ。

### Precision重視

```text
陽性判定を出したなら、
できるだけ外したくない
```

例えば、

```text
広告のターゲティング
```

など。

---

### Recall重視

```text
本当に陽性のものを、
できるだけ見逃したくない
```

例えば、

```text
病気のスクリーニング
異常検知
不正検知
```

などでは、Recallが重要になる場合があります。

---

# 💻 実習7：F1

PrecisionとRecallを一つにまとめたい場合、

**F1 score**

があります。

```python
from sklearn.metrics import f1_score

f1 = f1_score(
    y_test,
    pred
)

print(f1)
```

F1は、

[
F1 =
2
\frac{Precision \times Recall}
{Precision + Recall}
]

です。

PrecisionとRecallの**調和平均**です。

どちらか一方だけが極端に低いと、F1も低くなります。

---

# 💻 実習8：classification_report

毎回全部書くのは大変なので、

```python
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred
    )
)
```

とすると、

```text
precision
recall
f1-score
support
```

をまとめて確認できます。

これは実務でも非常によく使います。

---

# 📖 3. 「予測ラベル」だけでは情報が足りない

ここで、第14回を思い出してください。

```python
model.predict_proba(X)
```

で、

```text
クラスA  0.51
クラスB  0.49
```

のような確率を取得できました。

`predict()` は、

> 一番確率の高いクラスを選んでいる

だけです。

つまり、

```text
0.51 vs 0.49
```

と

```text
0.99 vs 0.01
```

は、どちらも同じ「クラスA」という予測になります。

でも、**確信度は全然違います。**

そこで確率そのものを評価する方法があります。

---

# 📖 4. ROC-AUC

その代表が、

**ROC-AUC**

です。

まずROCは、

```text
threshold（判定閾値）
```

を動かしながら、

```text
True Positive Rate
False Positive Rate
```

の関係を見るものです。

---

## Thresholdとは？

例えば、

```text
病気確率 > 0.5
```

なら病気と判断するとします。

でも、

```text
0.5
```

を

```text
0.3
```

にしたら、

より多くの人を陽性と判定するようになります。

逆に、

```text
0.8
```

にすれば、

かなり確信があるものだけ陽性にします。

つまり、

> **モデルが出した確率を、どの閾値で分類するか**

によってPrecisionやRecallなどが変化します。

---

# 💻 実習9：ROC-AUC

```python
from sklearn.metrics import roc_auc_score

prob = model.predict_proba(
    X_test
)[:, 1]
```

そして、

```python
auc = roc_auc_score(
    y_test,
    prob
)

print(auc)
```

---

# 🧠 ROC-AUCの直感

かなりざっくり言えば、

> **陽性のデータに高いスコアを、陰性のデータに低いスコアを付ける能力**

を見ています。

一般に、

```text
0.5
```

付近ならランダムな識別に近く、

```text
1.0
```

に近いほど良い識別能力を示します。

ただし、ROC-AUCも万能ではありません。

特に極端なクラス不均衡では、Precision-Recall系の評価がより有用になることがあります。

---

# 📖 5. `cross_validate()` を使う

ここまで、

```python
cross_val_score()
```

を使ってきました。

でもモデル比較では、

```text
Accuracy
Precision
Recall
F1
```

など複数の指標を一度に見たいことがあります。

そこで、

```python
cross_validate()
```

を使います。

---

# 💻 実習10：複数指標を一気に評価

```python
from sklearn.model_selection import cross_validate

scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]
```

実行。

```python
results = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring=scoring
)
```

結果を見る。

```python
print(
    results.keys()
)
```

---

## 平均を見る

```python
for metric in scoring:

    scores = results[
        "test_" + metric
    ]

    print(
        metric,
        scores.mean()
    )
```

これで、

```text
accuracy
precision
recall
f1
roc_auc
```

を同じ5-fold CV条件で比較できます。

---

# 💻 実習11：Random Forestも評価

ここまで来たので、モデルを比較します。

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
```

同じ評価。

```python
rf_results = cross_validate(
    rf,
    X,
    y,
    cv=5,
    scoring=scoring
)
```

---

# 💻 実習12：Gradient Boostingも評価

```python
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
```

```python
gb_results = cross_validate(
    gb,
    X,
    y,
    cv=5,
    scoring=scoring
)
```

---

# 🧪 実験：3モデルを比較

3つを、

```text
Logistic Regression
Random Forest
Gradient Boosting
```

として、

```text
Accuracy
Precision
Recall
F1
ROC-AUC
```

を比較してみましょう。

例えば、

```python
models = {
    "Logistic": model,
    "Random Forest": rf,
    "Gradient Boosting": gb
}
```

そして、

```python
for name, model in models.items():

    results = cross_validate(
        model,
        X,
        y,
        cv=5,
        scoring=scoring
    )

    print(
        "\n",
        name
    )

    for metric in scoring:

        score = results[
            "test_" + metric
        ].mean()

        print(
            metric,
            score
        )
```

---

# 🧠 ここで「AI開発者の視点」

例えば結果が、

```text
                    Accuracy   Recall   Precision
Logistic Regression    0.97      0.98      0.95

Random Forest          0.96      0.97      0.94

Gradient Boosting      0.97      0.96      0.98
```

だったとします。

どれが一番良いでしょう？

**答えは「目的による」です。**

---

## 病気を見逃したくない

なら、

```text
Recall
```

を重視する可能性があります。

---

## 誤検知を減らしたい

なら、

```text
Precision
```

を重視する可能性があります。

---

## 両方をバランスよく見たい

なら、

```text
F1
```

が候補になります。

---

## 順位付け能力を見たい

なら、

```text
ROC-AUC
```

が候補になります。

---

# 🎯 今日の核心

ここで重要なのは、

> **評価指標は単なる「採点方法」ではなく、AIシステムが何を重視するかを表す設計要素**

だということです。

例えば、

```text
病気を見逃すコスト
```

と

```text
健康な人を病気と判定するコスト
```

が同じとは限りません。

したがって、

```text
モデル選択
```

の前に、

```text
何を失敗と呼ぶのか？
```

を考える必要があります。

---

# ✍️ 演習

## 問1

次の状況では、Accuracyだけで評価するのが危険なのはなぜでしょう？

```text
陽性：1%
陰性：99%
```

---

## 問2

次を説明してください。

```text
Precision
Recall
F1
```

それぞれ、

> **何を重視した指標なのか**

を自分の言葉で説明します。

---

## 問3

次のケースでは、PrecisionとRecallのどちらをより重視したいでしょう？

### A

重大な病気のスクリーニング

### B

スパムメール判定

### C

不正アクセス検知

※唯一の正解を暗記する問題ではありません。
「なぜそう考えたか」を説明することが目的です。

---

## 問4

`predict()` と `predict_proba()` の違いを説明してください。

---

## 問5

`cross_validate()` を使って、

```text
accuracy
precision
recall
f1
roc_auc
```

を5-fold CVで評価してください。

---

# 👾 ボス戦：モデル選択会議

今日の本丸です。

次の3モデルについて、

```text
Logistic Regression
Random Forest
Gradient Boosting
```

を同じ5-fold CV条件で評価してください。

評価指標は、

```text
Accuracy
Precision
Recall
F1
ROC-AUC
```

です。

そのうえで、

> **「この問題ではどのモデルを採用するか」**

を決めてください。

ただし、

**「Accuracyが一番高かったから」だけでは不合格。**

最低でも、

```text
1. 何を重視するか
2. どの指標を見るか
3. そのモデルを選んだ理由
```

を説明してください。

---

# 🌱 今日のまとめ

今日までの内容が、ここでかなり重要な形にまとまりました。

これまでは、

```text
モデルを作る
 ↓
Accuracyを見る
```

というところから始まりました。

今は、

```text
問題設定
 ↓
何を失敗とするか
 ↓
評価指標を決める
 ↓
候補モデルを作る
 ↓
Cross Validation
 ↓
複数指標で比較
 ↓
モデル選択
 ↓
最終テスト
```

という流れになっています。

これが、**機械学習を「ただモデルを動かす作業」から「AIシステムを設計する作業」へ変えるポイント**です。

---

# 🧭 AI工学101・現在地

ここまでのscikit-learn編を俯瞰すると、

```text
NumPy
 │
 ├─ 配列・ベクトル
 ├─ 行列演算
 └─ データ操作
       ↓
scikit-learn
 │
 ├─ 回帰
 ├─ 分類
 ├─ 前処理
 ├─ 特徴量
 ├─ Pipeline
 ├─ 評価指標
 ├─ Cross Validation
 ├─ Grid Search
 ├─ 汎化
 ├─ 過学習
 ├─ Decision Tree
 ├─ Random Forest
 ├─ Gradient Boosting
 └─ モデル比較・評価設計 ★
```

かなり「機械学習の基礎体力」らしくなってきた。

そして、ここで一つ大事な変化があります。

**「どのモデルが一番強いか」ではなく、「この問題では何を最適化すべきか？」と考え始めた。**

これはPyTorchに進んでも、ニューラルネットに進んでも、その先の研究に進んでもずっと残る考え方です。

---

# 🔜 第25回

## 不均衡データと分類閾値：Recallを上げると何が起こる？

次回は今日の話をさらに実践的にします。

テーマは、

* クラス不均衡
* `class_weight`
* Decision Threshold
* Precision–Recall trade-off
* ROC曲線とPR曲線
* 「0.5で分類する」という前提を疑う

です。

特に、

```python
predict()
```

が暗黙に使っている**分類閾値**を自分で動かして、

```text
Recallを上げる
↓
False Positiveが増える
↓
Precisionが下がる
```

というトレードオフを実際に観察します。

ここまで来ると、**「モデルを学習させる」だけでなく、「モデルをどう運用するか」まで設計する**段階に入ります。